In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("DataLakeIngestion") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

In [3]:
# Configuración recomendada: tamaño de bloque/fichero ~128 MB 
# (esto afecta a cómo Spark divide los datos al escribir) 
spark.conf.set("spark.sql.files.maxPartitionBytes", 134217728)  # 128 MB 
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")  # típico en Data Lake
# Configuración recomendada: tamaño de bloque/fichero ~128 MB 
# (esto afecta a cómo Spark divide los datos al escribir)
spark.conf.set("spark.sql.files.maxPartitionBytes", 134217728)  # 128 MB
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")  # típico en Data Lake

In [4]:
# Ruta base de tu Data Lake 
base_path = "./datalake" 
#Ruta base del Origen 
base_origen = "./Origen" 

In [5]:
# Leer CSVs 
df_articles = spark.read.option("header", True).csv(f"{base_origen}/articles.csv") 
df_customers = spark.read.option("header", True).csv(f"{base_origen}/customers.csv") 
# Leer Parquet 
df_transactions = spark.read.parquet(f"{base_origen}/transactions.parquet")

In [6]:
# Guardar cada dataset en su carpeta 
df_articles.write.mode("overwrite").parquet(f"{base_path}/articles") 
df_customers.write.mode("overwrite").parquet(f"{base_path}/customers") 
df_transactions.write.mode("overwrite").parquet(f"{base_path}/transactions")